In [ ]:
import requests
import time
import json

# ================= 설정 섹션 =================
# 발급받은 실제 API 키를 입력하세요.
API_KEY = "8bf63e2555c4a687a49f4cf0145f8aa0"
MODEL_AIR = "urn:air:sdxl:checkpoint:civitai:101055@128078" # 기본 SDXL 모델 예시

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

def generate_image(prompt, negative_prompt=""):
    # 1. 이미지 생성 요청 (Job 생성)
    url = "https://orchestration.civitai.com/v1/consumer/jobs"

    payload = {
        "model": MODEL_AIR,
        "params": {
            "prompt": prompt,
            "negativePrompt": negative_prompt,
            "scheduler": "EulerA",
            "steps": 25,
            "cfgScale": 7,
            "width": 1024,
            "height": 1024,
            "quantity": 1
        }
    }

    print(f"🚀 이미지 생성 요청 중... (Prompt: {prompt})")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        print(f"❌ 요청 실패: {response.status_code}")
        print(response.text)
        return None

    job_data = response.json()
    job_id = job_data.get("jobId")
    print(f"✅ 작업 생성 완료! Job ID: {job_id} (예상 비용: {job_data.get('cost')} Buzz)")

    # 2. 작업 상태 폴링 (Polling)
    poll_url = f"https://orchestration.civitai.com/v1/consumer/jobs/{job_id}"

    while True:
        poll_response = requests.get(poll_url, headers=headers)
        status_data = poll_response.json()

        # 구조에 따라 job['status'] 또는 status_data['status'] 확인
        status = status_data.get("status") or status_data.get("job", {}).get("status")

        if status == "Completed":
            # 결과물 URL 추출
            result = status_data.get("result") or status_data.get("job", {}).get("result")
            # Civitai 응답 구조에 따라 'blobUrl' 필드 확인
            image_url = result.get("blobUrl") if result else None
            print(f"\n✨ 생성 완료! 이미지 URL: {image_url}")
            return image_url
        elif status == "Failed":
            print("\n❌ 이미지 생성 실패")
            return None
        else:
            print(f"⏳ 대기 중... 현재 상태: {status}", end="\r")
            time.sleep(3) # 3초 간격으로 확인

# ================= 테스트 실행 =================
if __name__ == "__main__":
    my_prompt = "A high-tech futuristic laboratory, cinematic lighting, 8k resolution, highly detailed"
    image_url = generate_image(my_prompt)

    if image_url:
        print(f"\n브라우저에서 확인하세요: {image_url}")